# Démo 1 — Ajouter un artiste au catalogue

Tu entres un nom d'artiste ; le notebook le cherche sur **ChartMetric** (client
cache-first : un artiste déjà récupéré ne rappelle pas l'API), vérifie que son
**score de popularité Spotify est entre 40 et 80** (ni trop niche, ni superstar —
cohérent avec la fourchette calibrée du simulateur), puis :

- l'ajoute (ou le met à jour) dans `data/upsert/artists.csv` ;
- ajoute son genre à `data/upsert/genres.csv` **s'il est nouveau** ;
- affiche les salles de `data/upsert/venues.csv` les plus cohérentes avec sa
  popularité (informatif — les salles sont un référentiel indépendant de
  l'artiste, ce notebook ne les modifie pas).

**Important** : contrairement aux notebooks 03/04 (`rebuild_artists_csv` /
`rebuild_genres_csv`, qui régénèrent *tout le fichier* depuis le cache
ChartMetric), ce notebook fait une **mise à jour ciblée d'une seule ligne** —
un rebuild complet réintroduirait dans `artists.csv` des artistes présents dans
le cache brut mais volontairement retirés du catalogue officiel.

> Prérequis : `CHARTMETRIC_REFRESH_TOKEN` dans le `.env` à la racine du projet
> (cf. `.env.example`).

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../env"))

import numpy as np
import pandas as pd

from chartmetric import (
    ChartmetricClient, CsvStore, fetch_artist,
    _pick_genre, _age_distribution, _engagement_to_genre_params,
)

client = ChartmetricClient()      # lit CHARTMETRIC_REFRESH_TOKEN (env ou .env)
store = CsvStore()                # cache data/raw/chartmetric/
print("Client ChartMetric prêt.")

Client ChartMetric prêt.


In [2]:
# ============================ PARAMÈTRE ======================================
ARTIST_NAME = "Sabaton"          # <- nom de l'artiste à chercher / ajouter
# ===========================================================================
POP_MIN, POP_MAX = 40, 80        # fourchette de popularité Spotify (0-100) acceptée
ARTISTS_CSV = "../data/upsert/artists.csv"
GENRES_CSV = "../data/upsert/genres.csv"

In [3]:
# --- Recherche + jauge de popularité -----------------------------------------
cm_id = fetch_artist(client, ARTIST_NAME, store=store)
if cm_id is None:
    raise SystemExit(f"Artiste introuvable sur ChartMetric : {ARTIST_NAME!r}")

meta = store.load("artist_metadata")
m = meta.loc[pd.to_numeric(meta["cm_id"], errors="coerce") == cm_id].iloc[-1]

def _val(row, *names, default=0):
    for n in names:
        if n in row and pd.notna(row[n]):
            return row[n]
    return default

name_resolved = str(_val(m, "name", default=ARTIST_NAME))
popularity = float(_val(m, "cm_statistics_sp_popularity", default=float("nan")))
listeners = int(float(_val(m, "cm_statistics_sp_monthly_listeners")))
followers = int(float(_val(m, "cm_statistics_sp_followers")))

print(f"Artiste résolu     : {name_resolved}  (cm_id={cm_id})")
print(f"Popularité Spotify : {popularity:.0f} / 100")
print(f"Auditeurs mensuels : {listeners:,}".replace(",", "'"))
print(f"Abonnés            : {followers:,}".replace(",", "'"))

if popularity != popularity:                      # NaN : ChartMetric n'a pas fourni la donnée
    DANS_LA_FOURCHETTE = False
    print("\n-> popularité indisponible pour cet artiste : impossible de statuer, NON ajouté.")
elif POP_MIN < popularity < POP_MAX:
    DANS_LA_FOURCHETTE = True
    print(f"\n-> dans la fourchette ({POP_MIN}-{POP_MAX}) : ajout au catalogue.")
else:
    DANS_LA_FOURCHETTE = False
    trop = "trop niche" if popularity <= POP_MIN else "trop grand public"
    print(f"\n-> HORS fourchette ({POP_MIN}-{POP_MAX}, {trop}) : artiste NON ajouté au catalogue.")

- Sabaton (cm_id=210472)
Artiste résolu     : Sabaton  (cm_id=210472)
Popularité Spotify : 73 / 100
Auditeurs mensuels : 3'055'803
Abonnés            : 2'600'748

-> dans la fourchette (40-80) : ajout au catalogue.


In [4]:
# --- Mise à jour CIBLÉE de artists.csv (une seule ligne) + genres.csv --------
if DANS_LA_FOURCHETTE:
    known_genres = set(pd.read_csv(GENRES_CSV)["genre_name"].str.casefold())
    genre = _pick_genre(m, known_genres)
    ages = _age_distribution(cm_id, store, genre)

    new_row = {
        "artist_name": name_resolved,
        "primary_genre": genre,
        "global_monthly_listeners": listeners,
        "total_followers": followers,
        "spotify_popularity": popularity,
        "cm_artist_score": round(float(_val(m, "cm_artist_score", "cm_statistics_cm_artist_score")), 2),
        "age_13_17_pct": round(ages[0], 4), "age_18_24_pct": round(ages[1], 4),
        "age_25_34_pct": round(ages[2], 4), "age_35_44_pct": round(ages[3], 4),
        "age_45_plus_pct": round(ages[4], 4),
    }

    artists = pd.read_csv(ARTISTS_CSV)
    artists = artists[artists["artist_name"].str.casefold() != name_resolved.casefold()]  # remplace si déjà présent
    artists = pd.concat([artists, pd.DataFrame([new_row])], ignore_index=True).sort_values("artist_name")
    artists.to_csv(ARTISTS_CSV, index=False, encoding="utf-8-sig")
    print(f"{ARTISTS_CSV} : {len(artists)} artistes (ligne '{name_resolved}' ajoutée/mise à jour).")

    genres = pd.read_csv(GENRES_CSV)
    if genre.casefold() not in set(genres["genre_name"].str.casefold()):
        ratio = followers / max(1, listeners)                    # proxy fidélité pour CE nouvel artiste
        conv, mob = _engagement_to_genre_params(ratio)
        genres = pd.concat([genres, pd.DataFrame([{
            "genre_name": genre, "base_conversion_rate_pct": conv, "mobility_factor": mob,
        }])], ignore_index=True).sort_values("genre_name")
        genres.to_csv(GENRES_CSV, index=False, encoding="utf-8-sig")
        print(f"{GENRES_CSV} : nouveau genre ajouté -> {genre} (conversion={conv}, mobilité={mob}).")
    else:
        print(f"{GENRES_CSV} : genre '{genre}' déjà connu -> inchangé.")

    print("\nLigne artists.csv :")
    print(pd.DataFrame([new_row]).to_string(index=False))

../data/upsert/artists.csv : 13 artistes (ligne 'Sabaton' ajoutée/mise à jour).
../data/upsert/genres.csv : genre 'Metal' déjà connu -> inchangé.

Ligne artists.csv :
artist_name primary_genre  global_monthly_listeners  total_followers  spotify_popularity  cm_artist_score  age_13_17_pct  age_18_24_pct  age_25_34_pct  age_35_44_pct  age_45_plus_pct
    Sabaton         Metal                   3055803          2600748                73.0            83.66         0.0334          0.298         0.4342         0.1895            0.045


In [5]:
# --- Salles les plus cohérentes avec la popularité de l'artiste (informatif) --
# N'écrit rien dans venues.csv : les salles sont un référentiel indépendant de
# l'artiste (même formule que TicketMarketModel.sample_for_popularity).
if DANS_LA_FOURCHETTE:
    import importlib
    import refdata
    importlib.reload(refdata)                  # relit artists.csv (_read_csv est en lru_cache)
    from refdata import ArtistProfile

    artist = ArtistProfile.by_name(name_resolved)
    pop_idx = artist.popularity_index
    venues = pd.read_csv("../data/upsert/venues.csv")
    target = 400.0 * pop_idx
    venues = venues.assign(ecart=np.abs(np.log(venues["max_capacity"]) - np.log(target)))
    venues = venues.sort_values("ecart").drop(columns="ecart")

    print(f"Indice de popularité interne (1-10) : {pop_idx:.1f}  "
          f"(capacité cible ~ {target:,.0f} places)".replace(",", "'"))
    print("\nSalles les plus cohérentes :")
    print(venues.head(3).to_string(index=False))
else:
    print("(artiste non ajouté — pas de salle à suggérer)")

Indice de popularité interne (1-10) : 6.3  (capacité cible ~ 2'520 places)

Salles les plus cohérentes :
           venue_name     city  max_capacity  fan_cost_index_chf
        Victoria Hall   Geneve          1600                25.0
Auditorium Stravinski Montreux          4000                30.0
     Z7 Konzertfabrik Pratteln          1500                40.0
